# Демонстрация работы аудио-кодека SoundStream

Этот ноутбук демонстрирует основные этапы работы нейронного аудио-кодека и визуализирует процесс сжатия и восстановление входного аудиосигнала.

**Как пользоваться:**
1. Запусти все ячейки по порядку (Runtime $\rightarrow$ Run all).
2. В ячейке с переменной `AUDIO_URL` подставь свою прямую ссылку на WAV-файл (по умолчанию стоит пример из LJ Speech).
3. После прогона в самом низу появятся два аудиоплеера: оригинал и реконструкция.

## 1. Загрузка репозитория и зависимостей

Клонируем репозиторий и устанавливаем зависимости из requirements.txt (как описано в README).

In [ ]:
!git clone -b soundstream-gan https://github.com/DommeUse/pytorch_project_template.git
%cd pytorch_project_template
!pip install -r requirements.txt

## 2. Загрузка модели

Скачиваем предобученный чекпоинт SoundStream, создаём модель и подгружаем веса.

In [2]:
# Веса модели лежат на Google Drive по публичной ссылке, поэтому скачаем библиотеку для взаимодействия с диском
!pip install -q gdown

In [3]:
# Подгрузим файл с весами
import gdown

CHECKPOINT_GDRIVE_ID = '1s0WCqekMwSmOkwMP8Cibc1LVSbUBi6oC'
CHECKPOINT_PATH = 'checkpoint.pth'

gdown.download(id = CHECKPOINT_GDRIVE_ID, output = CHECKPOINT_PATH, quiet = False)

Downloading...
From (original): https://drive.google.com/uc?id=1s0WCqekMwSmOkwMP8Cibc1LVSbUBi6oC
From (redirected): https://drive.google.com/uc?id=1s0WCqekMwSmOkwMP8Cibc1LVSbUBi6oC&confirm=t&uuid=e3a83322-cf66-44ad-8d26-5d900693a401
To: /content/pytorch_project_template/checkpoint.pth
100%|██████████| 378M/378M [00:05<00:00, 74.8MB/s]


'checkpoint.pth'

In [4]:
# Создаём модель с теми же гиперпараметрами, что и при обучении
import torch
from pathlib import Path
from src.model import SoundStream
from src.trainer import FileInferencer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device = {device}')

model = SoundStream(
    encoder_channels = 32,
    target_channels = 128,
    n_quantizers = 8,
    codebook_size = 1024,
).to(device)

# В чекпойнте от трейнера веса генератора лежат под ключом 'state_dict'.
checkpoint = torch.load(CHECKPOINT_PATH, map_location = device, weights_only = False)
state_dict = checkpoint.get('state_dict', checkpoint)
model.load_state_dict(state_dict, strict = True)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded SoundStream: {n_params / 1e6:.2f}M params')

# Инференсер внутри делает обработку входного аудио, поэтому вставляем в него сырое аудио
inferencer = FileInferencer(
    model = model,
    sample_rate = 16000,
    device = device,
    save_path = Path('demo_outputs') # тут вы можете вписать свой путь сохранения обработанных аудио
)

device = cpu
Loaded SoundStream: 8.27M params


## 3. Сжатие и восстановление пользовательского аудио

Подставь свою прямую ссылку на WAV в `AUDIO_URL` ниже. Ячейка скачает файл, прогонит его через обученный кодек и проиграет оригинал и реконструкцию.

In [5]:
# Сюда вам нужно вставить вашу ссылку на аудиофайл в формате .wav
AUDIO_URL = 'https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav'

In [6]:
# Скачиваем файл по ссылке.
import urllib.request

INPUT_PATH = 'input.wav'
urllib.request.urlretrieve(AUDIO_URL, INPUT_PATH)
print(f'Downloaded {AUDIO_URL} -> {INPUT_PATH}')

Downloaded https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav -> input.wav


In [7]:
# Прогоняем через кодек. FileInferencer возвращает numpy-массив реконструкции
# на 16 kHz и параллельно пишет файл в demo_outputs/reconstructed.wav.
reconstructed = inferencer(INPUT_PATH, output_name = 'reconstructed.wav')
print(f'reconstructed shape: {reconstructed.shape}, dtype: {reconstructed.dtype}')

reconstructed shape: (134347,), dtype: float32


In [8]:
# Воспроизводим оригинал и реконструкцию.
import IPython.display as ipd

print('Оригинал:')
ipd.display(ipd.Audio(INPUT_PATH))

print('Реконструкция SoundStream (16 kHz):')
ipd.display(ipd.Audio(reconstructed, rate = 16000))

Оригинал:


Реконструкция SoundStream (16 kHz):
